# 01 - Review collection

Collection ran once, on 2026-07-19, and its output is the immutable raw file `data/raw/reviews_dataset.csv`. This notebook documents how that file was produced and lets the corpus be regenerated; the later notebooks never modify it.

Two public, unauthenticated endpoints are used:

- **Google Play** through `google-play-scraper`, UAE storefront, queried per language with Arabic first and a separate budget per language so the Arabic pass is never starved by the larger English volume.
- **Apple App Store** through the public RSS customer-reviews feed (`itunes.apple.com/ae/rss/customerreviews/...`), ten pages per app.

The app list (`APP_LIST` in `src/config.py`, 155 identifiers across federal entities, the seven emirates and semi-government bodies) was resolved by searching both stores by entity name and keeping only government publishers. Requests are rate limited, the run is resumable through a checkpoint file, and every review is deduplicated on its store id and on its normalised text. The development version of this scraper, with the identifier-resolution searches, is kept in `notebooks/legacy/00_scrape_dev.ipynb`.

**Set `RUN_SCRAPE = True` to collect again.** That overwrites nothing: the scraper appends to the raw file and skips apps already recorded in `data/raw/scrape_checkpoint.txt`.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# make src importable when the kernel starts inside notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid", context="notebook")
from src import config
from src.io import ensure_dirs, save_table, update_metrics, read_metrics
ensure_dirs()

In [ ]:
from src import scraping
from src.config import APP_LIST, RAW_FILE

RUN_SCRAPE = False
print(f"{len(APP_LIST)} apps in the collection frame; "
      f"{sum(1 for a in APP_LIST if a[1])} on Google Play, {sum(1 for a in APP_LIST if a[2])} on the App Store")
print("raw file:", RAW_FILE, "exists" if RAW_FILE.exists() else "missing")

In [ ]:
if RUN_SCRAPE:
    scraping.main(APP_LIST)
else:
    print("skipped - raw file already collected")

## What was collected

In [ ]:
raw = pd.read_csv(RAW_FILE, encoding="utf-8-sig", dtype=str)
print(f"{len(raw):,} reviews, {raw.app_name.nunique()} apps")
print(raw.platform.value_counts().to_dict())
print(raw.language.value_counts().to_dict())
print(raw.satisfaction_label.value_counts().to_dict())
raw.groupby("app_name").size().sort_values(ascending=False).head(15)